In [1]:
import numpy as np

In [3]:
class Detector:
    def __init__(self, R=2, N=40, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a

        self.pm_positions = self.generate_positions()

    def generate_positions(self):
        """Равномерное распределение фотоумножителей по сфере."""
        positions = []
        golden_angle = np.pi * (3 - np.sqrt(5))

        for i in range(self.N):
            z = 1 - 2 * (i + 0.5) / self.N
            theta = golden_angle * i
            rho = np.sqrt(1 - z**2)

            x = self.R * rho * np.cos(theta)
            y = self.R * rho * np.sin(theta)
            z = self.R * z

            positions.append([x, y, z])

        return np.array(positions)

    def is_on_detector(self, point, tol=1e-6):
        return abs(np.linalg.norm(point) - self.R) < tol

    def belongs_to_pm(self, point):
        """Проверяет, попадает ли точка в какой-либо фотоумножитель."""
        point = np.asarray(point)

        if not self.is_on_detector(point):
            return False

        cos_angles = self.pm_positions @ point / self.R**2
        cos_angles = np.clip(cos_angles, -1, 1)

        distances = self.R * np.arccos(cos_angles)

        return np.any(distances <= self.r)

    def find_photomultiplier(self, source, direction):
        """
        Определяет, в какой фотоумножитель попал фотон.
        Возвращает индекс фотоумножителя или None.
        """
        source = np.asarray(source)
        direction = np.asarray(direction)

        # Находим пересечение траектории со сферой
        dot = np.dot(source, direction)

        t = -dot + np.sqrt(
            dot**2 + self.R**2 - np.dot(source, source)
        )

        hit_point = source + t * direction

        # Проверяем расстояния до всех фотоумножителей
        cos_angles = self.pm_positions @ hit_point / self.R**2
        cos_angles = np.clip(cos_angles, -1, 1)

        distances = self.R * np.arccos(cos_angles)

        # Индексы фотоумножителей, на которые попал фотон
        hits = np.where(distances <= self.r)[0]

        if len(hits) == 0:
            return None

        return hits[0]

    def generate_photoelectrons(self, n_photons):
        """Разыгрывает количество фотоэлектронов."""
        return np.random.poisson(self.a * n_photons)


class Source:
    def __init__(self, Y=10000):
        self.Y = Y

    def generate_directions(self, E):
        """Генерирует направления движения фотонов."""
        N = int(self.Y * E)

        phi = np.random.uniform(0, 2 * np.pi, N)
        cos_theta = np.random.uniform(-1, 1, N)
        sin_theta = np.sqrt(1 - cos_theta**2)

        directions = np.column_stack((
            sin_theta * np.cos(phi),
            sin_theta * np.sin(phi),
            cos_theta
        ))

        return directions


In [13]:
# Моделирование одного события
detector = Detector()
source = Source()

E = 1  # МэВ
source_position = np.array([0., 0., 0.])

directions = source.generate_directions(E)

# Считаем количество фотонов, попавших в каждый фотоумножитель
photons_per_pm = np.zeros(detector.N, dtype=int)

for direction in directions:
    pm_index = detector.find_photomultiplier(
        source_position, direction
    )

    if pm_index is not None:
        photons_per_pm[pm_index] += 1

# Разыгрываем количество фотоэлектронов
photoelectrons = np.array([
    detector.generate_photoelectrons(n)
    for n in photons_per_pm
])

# Полный собранный заряд в единицах элементарного заряда
Q_simulated = np.sum(photoelectrons)

# Теоретическое ожидание по формуле из задания
Q_expected = detector.a * source.Y * E * (
    detector.N * detector.r**2 / (4 * detector.R**2)
)

print("Количество испущенных фотонов:", len(directions))
print("Количество попавших фотонов:", np.sum(photons_per_pm))
print("Количество фотоэлектронов:", Q_simulated)
print("Ожидаемое значение Q:", Q_expected)
print("Относительное отклонение:",
      abs(Q_simulated - Q_expected) / Q_expected * 100, "%")

Количество испущенных фотонов: 10000
Количество попавших фотонов: 248
Количество фотоэлектронов: 129
Ожидаемое значение Q: 125.00000000000003
Относительное отклонение: 3.1999999999999766 %


In [5]:
results = []

for _ in range(1000):
    directions = source.generate_directions(E=1)
    photons_per_pm = np.zeros(detector.N, dtype=int)

    for direction in directions:
        pm_index = detector.find_photomultiplier(
            np.zeros(3), direction
        )

        if pm_index is not None:
            photons_per_pm[pm_index] += 1

    photoelectrons = detector.generate_photoelectrons(
        np.sum(photons_per_pm)
    )

    results.append(photoelectrons)

print("Среднее:", np.mean(results))
print("Стандартное отклонение:", np.std(results))
print("Теоретическое ожидание:", 125)

Среднее: 125.223
Стандартное отклонение: 13.70938623717342
Теоретическое ожидание: 125


Теоретическое математическое ожидание составляет:

$$
\mathbb E[Q]=aYE\frac{Nr^2}{4R^2}=125.
$$

С учётом случайности как процесса регистрации фотонов, так и генерации фотоэлектронов, теоретическое стандартное отклонение составляет:

$$
\sigma_Q=\sqrt{250\cdot0.5\cdot1.5}\approx13.69.
$$

Полученные результаты согласуются с теоретическими оценками, что подтверждает корректность реализованной модели.